In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from retia_cognitionis.config import RAW_DATA_DIR, INTERIM_DATA_DIR

RAW = RAW_DATA_DIR / "ds000030"
na = ["n/a", "-9999"]

def load(name, cols):
    return pd.read_csv(f"{RAW}/phenotype/{name}.tsv", sep="\t", na_values=na)[["participant_id", *cols]]

df = (
    pd.read_csv(f"{RAW}/participants.tsv", sep="\t", na_values=na)
    .merge(load("acds_adult", ["adult_attention", "adult_hyperactivity", "adhd_c_subtype"]), on="participant_id", how="left")
    .merge(load("asrs", ["asrs_score"]), on="participant_id", how="left")
    .merge(load("sans", ["factor_attention", "factor_avolition"]), on="participant_id", how="left")
)
df = df[df.diagnosis.isin(["CONTROL", "SCHZ", "ADHD"])]
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_DATA_DIR / "phenotype.tsv", sep="\t", index=False)